In [60]:
from typing import List,Optional
import pydantic
from pydantic import BaseModel
import json
import random
from pydantic import ValidationError
from colorama import Fore
from haystack import component
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.generators.chat import OpenAIChatGenerator


In [61]:
class City(BaseModel):
    name : str
    country : str
    population : int

class Cities(BaseModel):
    cities : List[City]
    

In [62]:
@component
class OutputValidator:
    def __init__(self,pydantic_model:pydantic.BaseModel):
        self.pydantic_model = pydantic_model
        self.iteration_counter = 0

    @component.output_types(valid_replies=List[str] , invalid_replies=Optional[List[str]] , error_messages = Optional[str] )
    def run(self, replies : List[ChatMessage]):
        self.iteration_counter += 1
        try:
            output_dict = json.loads(replies[0].text)
            self.pydantic_model.model_validate(output_dict)
            print(Fore.GREEN + f"In Iteration number {self.iteration_counter} , Valid json from LLM - No need for looping {replies[0]}")
            return {"valid_replies" : replies}
        except (ValueError , ValidationError) as e :
            print(Fore.RED + f"In Iteration number {self.iteration_counter} , Invalid json from LLM - need for looping {replies[0]} , Error : {e}")
            return {"invalid_replies" : replies , "error_messages" : str(e)}
        
    

In [63]:
output_validator = OutputValidator( pydantic_model = Cities)

In [64]:
from haystack.dataclasses import ChatMessage

template = [ ChatMessage.from_user(
    """
Create a JSON object from the information present in this passage: {{passage}}.
Only use information that is present in the passage. Follow this JSON schema, but only return the actual instances without any additional schema definition:
{{schema}}
Make sure your response is a dict and not a list.
{% if invalid_replies and error_message %}
  You already created the following output in a previous attempt: {{invalid_replies}}
  However, this doesn't comply with the format requirements from above and triggered this Python exception: {{error_message}}
  Correct the output and try again. Just return the corrected output without any extra explanations.
{% endif %}
"""
)]

prompt_builder = ChatPromptBuilder(template=template)

In [65]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')
generator = OpenAIChatGenerator(model = "gpt-4o-mini")

In [66]:
from haystack import Pipeline
structured_output_validator = Pipeline(max_runs_per_component=5)
structured_output_validator.add_component("prompt_builder" , prompt_builder)
structured_output_validator.add_component("generator" , generator)
structured_output_validator.add_component("output_validator" , output_validator)

In [67]:
structured_output_validator.connect("prompt_builder.prompt","generator.messages")
structured_output_validator.connect("generator.replies","output_validator")
structured_output_validator.connect("output_validator.invalid_replies","prompt_builder.invalid_replies")
structured_output_validator.connect("output_validator.error_messages","prompt_builder.error_message")


🚅 Components
  - prompt_builder: ChatPromptBuilder
  - generator: OpenAIChatGenerator
  - output_validator: OutputValidator
🛤️ Connections
  - prompt_builder.prompt -> generator.messages (List[ChatMessage])
  - generator.replies -> output_validator.replies (List[ChatMessage])
  - output_validator.invalid_replies -> prompt_builder.invalid_replies (Optional[List[str]])
  - output_validator.error_messages -> prompt_builder.error_message (Optional[str])

In [68]:
structured_output_validator.draw("auto-correct-pipeline.png")

In [69]:
json_schema = Cities.model_json_schema()
passage = "Berlin is the capital of Germany. It has a population of 3,850,809. Paris, France's capital, has 2.161 million residents. Lisbon is the capital and the largest city of Portugal with the population of 504,718."

result = structured_output_validator.run({"prompt_builder" : {"passage": passage , "schema" :json_schema}})

In Iteration number 1 , Valid json from LLM - No need for looping ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text='{\n  "cities": [\n    {\n      "name": "Berlin",\n      "country": "Germany",\n      "population": 3850809\n    },\n    {\n      "name": "Paris",\n      "country": "France",\n      "population": 2161000\n    },\n    {\n      "name": "Lisbon",\n      "country": "Portugal",\n      "population": 504718\n    }\n  ]\n}')], _name=None, _meta={'model': 'gpt-4o-mini-2024-07-18', 'index': 0, 'finish_reason': 'stop', 'usage': {'completion_tokens': 91, 'prompt_tokens': 242, 'total_tokens': 333, 'completion_tokens_details': CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), 'prompt_tokens_details': PromptTokensDetails(audio_tokens=0, cached_tokens=0)}})


In [70]:
valid = (result["output_validator"]["valid_replies"][0].text)
print(json.loads(valid))

{'cities': [{'name': 'Berlin', 'country': 'Germany', 'population': 3850809}, {'name': 'Paris', 'country': 'France', 'population': 2161000}, {'name': 'Lisbon', 'country': 'Portugal', 'population': 504718}]}
